# 读出 PDM 区间与边缘密度

本例使用 0.1.1 候选源码，默认计算内置的合成 `P_ADO`。已有结果时，把 `result_dir` 改成 GUI 或 CLI 保存完成的结果目录，即包含 `manifest.json` 和 `density.npz` 的目录。`Path` 可使用绝对路径，也可使用相对当前 Notebook 的路径。

Run the bundled synthetic input or set `result_dir` to a completed GUI/CLI snapshot.

In [ ]:
%matplotlib inline
from pathlib import Path

from multipolepdm.io.export import load_result
from multipolepdm.io.input import example_data, job_from_dict
from multipolepdm.jobs.workers import compute_job
from multipolepdm.pdm.statistics import summarize_density
from multipolepdm.plotting import make_preview

result_dir = None  # Example: Path('pdm-results/my-result')
if result_dir is None:
    job = job_from_dict(example_data('P_ADO'))
    result, summary = compute_job(job)
else:
    result = load_result(result_dir)
    summary = summarize_density(result)

print(result.job.mode, result.job.label)
print('Source spin I =', result.response.reference_spin)
print('Finite-scan normalization =', result.normalization)
for warning in result.warnings:
    print(warning)

角空间先求蓝色68% HDI、橙色q16/q50/q84与灰色均值和标准差。Summary.tan_angle_statistics的结果显式写作tan{mode[arctan(δ)]}、tan{HDI68%[arctan(δ)]}、tan{mean[arctan(δ)]}，不是δ密度自身的峰、HDI或算术均值。tan_mean_std_endpoints只是一对转换端点，越界查看mean_bound_flags。Summary.delta_pdf保存真正的pδ=pξ/(1+δ²)，不是只换横轴；角HDI阈值不能直接用作δ密度的水平线。宽度统计直接在sigma/I密度上计算，I为源态自旋。

图题Full Range Marginalization; Constant Weight表示扫描全域边缘化与常数附加权重。观测图与参数图使用viridis，角轴显示π分数。

In [ ]:
for name, marginal in (
    ('arctan(delta) [rad]', summary.arctan_delta),
    ('sigma/I', summary.sigma_over_i),
):
    interval = marginal.intervals[0]
    print('\n' + name)
    print('ETI 68%  q16, q50, q84:', interval.eti[0], marginal.median, interval.eti[1])
    for number, segment in enumerate(interval.hdi_segments, start=1):
        print(f'HDI 68%  segment {number}:', segment.bounds,
              'mass =', segment.mass, 'mode ranges =', segment.mode_ranges)
    print('Total HDI mass:', sum(segment.mass for segment in interval.hdi_segments))
    print('Mean and mean +/- std endpoints:', marginal.mean, marginal.mean_bounds)
    if marginal.std is not None:
        print('SD:', marginal.std)
    if any(marginal.mean_bound_flags):
        print('Mean bound flags:', marginal.mean_bound_flags)

statistics = summary.tan_angle_statistics
print('tan{mode[arctan(δ)]}:', statistics.tan_mode)
print('tan{q50[arctan(δ)]}:', statistics.tan_median)
print('tan{mean[arctan(δ)]}:', statistics.tan_mean)
print('Transformed endpoints tan(mu +/- std):', statistics.tan_mean_std_endpoints)
for segment in statistics.intervals[0].tan_hdi_segments:
    print('tan{HDI68%[arctan(δ)]}:', segment.bounds, 'mass:', segment.mass)
    print('Local tan{mode[arctan(δ)]}:', segment.mode_ranges)
print('True delta PDF samples:', summary.delta_pdf.coordinate.size)
make_preview(result, summary).figure